In [3]:
"""
📘 2.1 从命中文本中提取纯段落（彻底去除头部与标签，包括前导空格）
📁 每个输入文件输出为一个新的纯净 TXT
"""
import os
import re
from pathlib import Path

DATA_ROOT = Path("Data")

# 可调参数：过滤掉长度过短的“段落”
MIN_PARAGRAPH_LEN = 20

def extract_clean_paragraphs(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # 标准化换行
    content = content.replace("\r\n", "\n").replace("\r", "\n")

    # === 1) 删除所有头部元信息行（整行删除）===
    header_pattern = re.compile(
        r"^\s*(?:"
        r"来源文件\s*:\s*.*"
        r"|Title\s*:\s*.*"
        r"|Journal\s*:\s*.*"
        r"|Date\s*:\s*.*"
        r"|段编号范围\s*:\s*.*"
        r"|段编号\s*:\s*.*"
        r"|材料关键词\s*:\s*.*"
        r"|力学关键词\s*:\s*.*"
        r"|降解关键词\s*:\s*.*"
        r")\s*$",
        flags=re.M
    )
    content = re.sub(header_pattern, "", content)

    # 某些文件可能包含分隔符一类的行，粗暴清理
    content = re.sub(r"^\s*={3,}\s*$", "", content, flags=re.M)

    # === 2) 删除标签行（含“段XX”）或仅标签 ===
    # 形如：
    #   [上下文段落] 段34
    #   [命中段落] 段35
    #   [前一段]
    #   [后一段]
    tag_line_pattern = re.compile(
        r"^\s*\[(?:前一段|后一段|命中段落|上下文段落)\]\s*(?:段\s*\d+)?\s*$",
        flags=re.M
    )
    content = re.sub(tag_line_pattern, "", content)

    # 兜底：若出现“段 34”单独一行，去掉
    content = re.sub(r"^\s*段\s*\d+\s*$", "", content, flags=re.M)

    # === 3) 合并多余空行，保证段落分割清晰 ===
    # 清理多余空格
    content = re.sub(r"[ \t]+\n", "\n", content)  # 去掉行尾空格
    # 合并 3 个以上的空行为 2 个
    content = re.sub(r"\n{3,}", "\n\n", content).strip()

    # === 4) 按空行分段，并过滤过短段落 ===
    paragraphs = re.split(r"\n\s*\n", content)
    paragraphs = [p.strip() for p in paragraphs if len(p.strip()) >= MIN_PARAGRAPH_LEN]

    return paragraphs


if __name__ == "__main__":
    input_folder = DATA_ROOT / "Wiley" / "hit_paragraphs"
    output_folder = DATA_ROOT / "Wiley" / "plain_text"
    os.makedirs(output_folder, exist_ok=True)

    print(f"🚀 开始处理文件夹：{input_folder}\n")

    counter = 0
    skip_counter = 0  # 新增：跳过计数器
    
    # 获取输入文件列表
    files = [f for f in os.listdir(input_folder) if f.lower().endswith(".txt")]
    
    for fname in files:
        fpath = os.path.join(input_folder, fname)
        out_path = os.path.join(output_folder, fname)

        # === 新增：核心跳过逻辑 ===
        if os.path.exists(out_path):
            skip_counter += 1
            # 如果文件很多，可以取消下面这行的注释来查看跳过详情
            # print(f"⏭️  跳过已处理文件: {fname}")
            continue
        # ========================

        try:
            paragraphs = extract_clean_paragraphs(fpath)
            if not paragraphs:
                # 注意：即便没有有效段落，也可以考虑生成一个空文件或记录
                # 否则下次运行还会尝试解析它。这里维持你原有的跳过逻辑。
                print(f"⚠️ {fname} 无有效段落，不输出。")
                continue

            clean_text = "\n\n".join(paragraphs)

            with open(out_path, "w", encoding="utf-8") as fw:
                fw.write(clean_text)

            counter += 1
            if counter % 10 == 0:
                print(f"🧩 正在处理... 已完成 {counter} 个新文件。")

        except Exception as e:
            print(f"❌ {fname} 处理失败: {e}")

    print(f"\n✅ 处理任务结束！")
    print(f"📊 统计：新处理 {counter} 个，跳过 {skip_counter} 个。")
    print(f"📁 输出目录：{output_folder}")

🚀 开始处理文件夹：D:\FXR\1111-HTML\Wiley\Hit段落

🧩 正在处理... 已完成 10 个新文件。
🧩 正在处理... 已完成 20 个新文件。
🧩 正在处理... 已完成 30 个新文件。
🧩 正在处理... 已完成 40 个新文件。
🧩 正在处理... 已完成 50 个新文件。
🧩 正在处理... 已完成 60 个新文件。
🧩 正在处理... 已完成 70 个新文件。
🧩 正在处理... 已完成 80 个新文件。
🧩 正在处理... 已完成 90 个新文件。
🧩 正在处理... 已完成 100 个新文件。
🧩 正在处理... 已完成 110 个新文件。
🧩 正在处理... 已完成 120 个新文件。
🧩 正在处理... 已完成 130 个新文件。
🧩 正在处理... 已完成 140 个新文件。
🧩 正在处理... 已完成 150 个新文件。
🧩 正在处理... 已完成 160 个新文件。
🧩 正在处理... 已完成 170 个新文件。
🧩 正在处理... 已完成 180 个新文件。
🧩 正在处理... 已完成 190 个新文件。
🧩 正在处理... 已完成 200 个新文件。
🧩 正在处理... 已完成 210 个新文件。
🧩 正在处理... 已完成 220 个新文件。
🧩 正在处理... 已完成 230 个新文件。
🧩 正在处理... 已完成 240 个新文件。
🧩 正在处理... 已完成 250 个新文件。
🧩 正在处理... 已完成 260 个新文件。
🧩 正在处理... 已完成 270 个新文件。
🧩 正在处理... 已完成 280 个新文件。
🧩 正在处理... 已完成 290 个新文件。
🧩 正在处理... 已完成 300 个新文件。
🧩 正在处理... 已完成 310 个新文件。
🧩 正在处理... 已完成 320 个新文件。
🧩 正在处理... 已完成 330 个新文件。

✅ 处理任务结束！
📊 统计：新处理 336 个，跳过 0 个。
📁 输出目录：D:\FXR\1111-HTML\Wiley\Hit段落\纯文本段落


In [5]:
"""
📘 2.234 (最终进阶版) 智能切片 + 断点续传 + 实时存档
功能：
1. 使用 LLM 自动提取 "潜在标题/关键词"
2. 自动跳过已处理的文件 (断点续传)
3. 边跑边存，防止报错丢失数据
4. 跑完自动合并为大文件
"""
import os
import time
import re
import random
import numpy as np
import pandas as pd

import os
from openai import OpenAI

# ====== ⚙️ 参数配置 ======
# 输入文件夹
input_folder = DATA_ROOT / "Wiley" / "plain_text"

# 输出总目录
output_base = DATA_ROOT / "Wiley" / "embeddings"

# 临时存放单个向量的文件夹 (跑完会自动合并，不要手动删)
npy_temp_folder = os.path.join(output_base, "temp_npy_chunks")
os.makedirs(npy_temp_folder, exist_ok=True)

# 最终输出文件路径
final_npy_path = os.path.join(output_base, "file_embeddings.npy")
final_csv_path = os.path.join(output_base, "file_metadata.csv")

CHUNK_SIZE = 600
CHUNK_OVERLAP = 150
MIN_TEXT_LEN = 30
SLEEP_SECONDS = 0.1

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.getenv(
        "LLM_BASE_URL",
        "https://llmapi.paratera.com/v1",
    ),
    timeout=30.0,
    max_retries=2,
)

TAG_MODEL = os.getenv("LLM_MODEL", "DeepSeek-V3.2")

# ====== 🛠️ 核心功能函数 ======

def load_full_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    text = text.replace("\ufeff", "").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def simple_text_splitter(text, chunk_size, overlap):
    if len(text) <= chunk_size: return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text): break
        start += (chunk_size - overlap)
    return chunks

def generate_semantic_tag(text_content):
    """调用 LLM 生成关键词，带重试机制"""
    preview = text_content[:800]
    prompt = f"""
        Please analyze the following academic text excerpt. Extract 3-5 core keywords or generate a concise title summarizing the content.
        Format Requirements: 
        - Output ONLY the keywords or title directly. 
        - Do not include any explanations.
        Text Excerpt:
        {preview}
    """    
    max_retries = 3
    base_wait = 2

    for attempt in range(max_retries):
        try:
            # 打印进度提示
            print(f"   🤖 请求 Topic ({attempt+1}/{max_retries})...", end="", flush=True)
            
            resp = client.chat.completions.create(
                model=TAG_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=60,
                timeout=15 # 单次请求超时
            )
            tag = resp.choices[0].message.content.strip()
            print(f" -> ✅ {tag[:15]}...") 
            return tag
            
        except Exception as e:
            print(f"\n   ❌ 失败: {e}")
            if "429" in str(e): # 限流等待
                time.sleep(base_wait * (2 ** attempt))
            else:
                time.sleep(1)
    
    return "学术文档" # 兜底返回值

def get_embedding_via_api(text: str):
    """获取向量，带简单的错误处理"""
    if not text: return None
    try:
        resp = client.embeddings.create(
            model="GLM-Embedding-2",
            input=text,
            timeout=20
        )
        return np.array(resp.data[0].embedding)
    except Exception as e:
        print(f"   ❌ Embedding 失败: {e}")
        return None

# ====== 🚀 主程序逻辑 ======

if __name__ == "__main__":
    txt_files = [fn for fn in os.listdir(input_folder) if fn.lower().endswith(".txt")]
    txt_files.sort()
    
    print(f"📂 扫描到 {len(txt_files)} 个文件。")

    # 1. 🔥 断点续传：读取已存在的 CSV，找出跑过的文件
    processed_files = set()
    if os.path.exists(final_csv_path):
        try:
            # 只读取 filename 列，速度快
            df_exist = pd.read_csv(final_csv_path, usecols=["filename"])
            processed_files = set(df_exist["filename"].unique())
            print(f"🔄 发现历史记录，将跳过 {len(processed_files)} 个已完成文件。")
        except Exception as e:
            print(f"⚠️ 读取历史 CSV 出错 ({e})，将重新开始。")
    
    # 2. 如果 CSV 不存在，先创建表头
    if not os.path.exists(final_csv_path):
        headers = ["filename", "auto_topic", "chunk_id", "npy_filename", "text", "text_for_emb"]
        pd.DataFrame(columns=headers).to_csv(final_csv_path, index=False, encoding="utf-8-sig")

    # 3. 开始循环处理
    for idx, fname in enumerate(txt_files, start=1):
        
        # 🔥 跳过检查
        if fname in processed_files:
            continue

        fpath = os.path.join(input_folder, fname)
        print(f"\n📄 [{idx}/{len(txt_files)}] 处理: {fname}")

        try:
            full_text = load_full_text(fpath)
            if len(full_text) < MIN_TEXT_LEN: 
                print("   ⚠️ 文本太短，跳过")
                continue

            # (A) 生成标签
            doc_topic = generate_semantic_tag(full_text)

            # (B) 切片
            chunks = simple_text_splitter(full_text, CHUNK_SIZE, CHUNK_OVERLAP)

            current_file_records = []

            # (C) 遍历切片 -> Embedding -> 存小文件
            for chunk_idx, chunk_text in enumerate(chunks):
                
                # 构造用于 Embedding 的文本
                text_for_embedding = f"File ID: {fname}\nTopic: {doc_topic}\nContent: {chunk_text}"
                
                # 获取向量
                emb = get_embedding_via_api(text_for_embedding)
                if emb is None: continue

                # 🔥 实时保存向量：每个切片存一个 .npy
                # 命名规则：文件名_切片ID.npy (防止冲突)
                safe_fname = fname.replace(".txt", "")
                npy_name = f"{safe_fname}_{chunk_idx}.npy"
                npy_full_path = os.path.join(npy_temp_folder, npy_name)
                
                np.save(npy_full_path, emb)

                # 准备元数据
                current_file_records.append({
                    "filename": fname,
                    "auto_topic": doc_topic,
                    "chunk_id": chunk_idx,
                    "npy_filename": npy_name, # 记录文件名，方便后续索引
                    "text": chunk_text,
                    "text_for_emb": text_for_embedding
                })
                
                time.sleep(SLEEP_SECONDS) # 避免 API 并发过高

            # (D) 🔥 实时保存 CSV：处理完一个文件，立刻写入
            # 这样就算下一个文件报错，当前文件的数据也已经落盘了
            if current_file_records:
                df_chunk = pd.DataFrame(current_file_records)
                # mode='a': 追加模式; header=False: 不重复写表头
                df_chunk.to_csv(final_csv_path, mode='a', header=False, index=False, encoding="utf-8-sig")
                print(f"   💾 已保存 {len(current_file_records)} 个切片数据")

        except Exception as e:
            print(f"❌ 严重错误: 处理文件 {fname} 时崩溃: {e}")
            # 捕获错误，确保循环不退出，继续跑下一个文件
            continue

    print("\n" + "="*50)
    print("✅ 所有文件处理完毕！正在执行最终合并...")

    # 4. 🔥 最终合并：把零散的 NPY 合并成一个大文件 (恢复原有逻辑)
    # 这一步是为了方便你后续读取，如果文件特别大(几个G)，建议就用分散的读取
    try:
        full_df = pd.read_csv(final_csv_path)
        all_embeddings = []
        valid_rows_indices = []

        print(f"📚 正在读取 {len(full_df)} 条记录并加载向量...")
        
        for i, row in full_df.iterrows():
            npy_name = row['npy_filename']
            npy_path = os.path.join(npy_temp_folder, npy_name)
            
            if os.path.exists(npy_path):
                vec = np.load(npy_path)
                all_embeddings.append(vec)
                valid_rows_indices.append(i)
            else:
                print(f"   ⚠️ 警告: 找不到向量文件 {npy_name}")

        if all_embeddings:
            big_matrix = np.vstack(all_embeddings)
            np.save(final_npy_path, big_matrix)
            print(f"🎉 成功！大矩阵已保存至: {final_npy_path}")
            print(f"📊 矩阵形状: {big_matrix.shape}")
        else:
            print("⚠️ 没有有效的向量数据可合并。")

    except Exception as e:
        print(f"❌ 合并阶段出错: {e}")
        print("💡 提示：你的单个 .npy 和 .csv 数据都是安全的，可以手动修复合并逻辑。")

📂 扫描到 336 个文件。

📄 [1/336] 处理: 101002_macp200900441_段105-107.txt
   🤖 请求 Topic (1/3)... -> ✅ Hydrolytic Degr...
   💾 已保存 1 个切片数据

📄 [2/336] 处理: 101002_macp200900441_段107-109.txt
   🤖 请求 Topic (1/3)... -> ✅ Nanomechanical ...
   💾 已保存 7 个切片数据

📄 [3/336] 处理: 101002_macp200900441_段112-114.txt
   🤖 请求 Topic (1/3)... -> ✅ Polyester Degra...
   💾 已保存 8 个切片数据

📄 [4/336] 处理: 101002_macp200900441_段34-36.txt
   🤖 请求 Topic (1/3)... -> ✅ Polyester Ureth...
   💾 已保存 8 个切片数据

📄 [5/336] 处理: 101002_macp200900441_段39-41.txt
   🤖 请求 Topic (1/3)... -> ✅ Degradation of ...
   💾 已保存 8 个切片数据

📄 [6/336] 处理: 101002_macp201000167_段27-29.txt
   🤖 请求 Topic (1/3)... -> ✅ HA-coated biode...
   💾 已保存 8 个切片数据

📄 [7/336] 处理: 101002_macp201000167_段53-55.txt
   🤖 请求 Topic (1/3)... -> ✅ Biodegradable H...
   💾 已保存 5 个切片数据

📄 [8/336] 处理: 101002_macp201000167_段66-68.txt
   🤖 请求 Topic (1/3)... -> ✅ Polymer Micelle...
   💾 已保存 6 个切片数据

📄 [9/336] 处理: 101002_macp201000167_段92-94.txt
   🤖 请求 Topic (1/3)... -> ✅ Biodegradable H.